### Sistema de Informações sobre Mortalidade (SIM) - Ministério da Saúde

SIM: arquivos nacionais anuais de mortalidade geral, 2020–2024.
Notebook: %pip install pandas pyarrow
          %run baixar_mortalidade_sim_2020_2024.py
Fonte: https://dadosabertos.saude.gov.br/dataset/sim
Downloads diretos HTTP, sem depender da API/catalogo durante a execucao.
Todas as causas e colunas preservadas como texto, inclusive codigos e datas.
Nao filtra COVID, nao deduplica e nao interpreta IDADE como anos.
Nao execute duas instancias na mesma pasta. Para atualizar a coleta, use uma
nova PASTA: arquivos concluidos sao reutilizados, nao sincronizados com a fonte.

In [4]:
import uuid
import csv
import hashlib
import io
import json
import time
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = None  # Nova pasta a cada execucao.
ANOS = list(range(2020, 2025))
LOTE = 50000
RAIZ = 'https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SIM/csv/'
URLS = {ano: RAIZ + f'Mortalidade_Geral_{ano}_csv.zip' for ano in range(2020, 2024)}
URLS[2024] = RAIZ + 'DO24OPEN_csv.zip'
for _ano in (2022, 2023):
    URLS[_ano] = RAIZ.replace('/csv/', '/xml/') + f'Mortalidade_Geral_{_ano}_xml.zip'

In [5]:
def salvar_json(p, obj):
    tmp = p.with_suffix('.json.part')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(p)


def sha256(p):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda: f.read(4 * 1024**2), b''): h.update(b)
    return h.hexdigest()


def baixar(url, destino):
    if destino.exists():
        with zipfile.ZipFile(destino) as z:
            if z.testzip() is not None: raise ValueError(f'ZIP corrompido: {destino}. Remova-o e execute novamente.')
        print('ZIP existente validado.', flush=True)
        return
    parcial = destino.with_suffix('.zip.part')
    for tentativa in range(1, 4):
        try:
            req = Request(url, headers={'User-Agent': 'Observatorio-COVID-Brasil/1.0', 'Accept-Encoding': 'identity'})
            with urlopen(req, timeout=90) as r, parcial.open('wb') as f:
                if r.status != 200: raise IOError(f'HTTP {r.status}; esperado arquivo completo (200).')
                total = int(r.headers.get('Content-Length', 0))
                recebido = 0
                ultimo = time.monotonic()
                while True:
                    bloco = r.read(1024**2)
                    if not bloco: break
                    f.write(bloco)
                    recebido += len(bloco)
                    if time.monotonic() - ultimo >= 3:
                        print(f'  Baixados {recebido / 1024**2:.1f} MB' + (f' / {total / 1024**2:.1f} MB' if total else ''), flush=True)
                        ultimo = time.monotonic()
            if recebido == 0 or (total and recebido != total): raise IOError('Download incompleto.')
            with zipfile.ZipFile(parcial) as z:
                if z.testzip() is not None: raise IOError('Falha na integridade do ZIP.')
            parcial.replace(destino)
            return
        except Exception as e:
            print(f'Tentativa {tentativa}/3: {e}', flush=True)
            if tentativa == 3: raise
            time.sleep(3 * tentativa)


def converter(origem, destino, ano):
    parcial = destino.with_suffix('.parquet.part')
    # Decodificacao estrita: se UTF-8 falhar, reinicia toda a conversao em Latin-1.
    for encoding in ('utf-8-sig', 'latin-1'):
        writer = None
        total = invalidas = outros_anos = 0
        membros = []
        try:
            with zipfile.ZipFile(origem) as z:
                membros = [n for n in z.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX/')]
                if not membros: raise ValueError('ZIP sem CSV.')
                esquema = None
                for nome in membros:
                    with z.open(nome) as raw, io.TextIOWrapper(raw, encoding=encoding, newline='') as f:
                        cabecalho = f.readline()
                    sep = csv.Sniffer().sniff(cabecalho, delimiters=';,\t').delimiter
                    colunas = next(csv.reader([cabecalho], delimiter=sep))
                    if len(colunas) != len(set(colunas)): raise ValueError('Colunas repetidas no CSV.')
                    if not {'DTOBITO', 'CAUSABAS'}.issubset(colunas): raise ValueError(f'Layout inesperado em {nome}: faltam DTOBITO/CAUSABAS.')
                    if esquema is None:
                        esquema = pa.schema([(c, pa.string()) for c in colunas])
                        writer = pq.ParquetWriter(parcial, esquema, compression='snappy')
                    elif colunas != esquema.names: raise ValueError('CSV com layouts diferentes no mesmo ZIP.')
                    with z.open(nome) as raw:
                        for df in pd.read_csv(raw, sep=sep, encoding=encoding, dtype='string', keep_default_na=False, chunksize=LOTE, on_bad_lines='error'):
                            datas = datas_obito(df['DTOBITO'])
                            invalidas += int(datas.isna().sum())
                            outros_anos += int((datas.notna() & datas.dt.year.ne(ano)).sum())
                            writer.write_table(pa.Table.from_pandas(df, schema=esquema, preserve_index=False))
                            total += len(df)
                            print(f'  {ano}: {total:,} registros convertidos', flush=True)
            writer.close()
            writer = None
            if total == 0: raise ValueError('Arquivo sem registros.')
            if pq.read_metadata(parcial).num_rows != total: raise ValueError('Contagem Parquet divergente.')
            parcial.replace(destino)
            return {'registros': total, 'colunas': len(esquema), 'encoding': encoding,
                    'csv_no_zip': membros, 'datas_invalidas_ou_ausentes': invalidas,
                    'datas_de_outro_ano': outros_anos}
        except UnicodeDecodeError:
            if encoding == 'latin-1': raise
            print('Reiniciando conversao com Latin-1...', flush=True)
        finally:
            if writer is not None: writer.close()


def datas_obito(serie):
    texto = serie.astype('string').str.strip()
    datas = pd.to_datetime(texto.where(texto.str.fullmatch(r'\d{7,8}', na=False)).str.zfill(8), format='%d%m%Y', errors='coerce')
    for formato in ('%d-%m-%Y', '%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d %H:%M:%S.%f'):
        datas = datas.fillna(pd.to_datetime(texto.where(datas.isna()), format=formato, errors='coerce'))
    return datas


def converter_xml(origem, destino, ano):
    """Le campos nomeados do XML em blocos; nao infere posicoes do CSV."""
    parcial = destino.with_suffix('.parquet.part')
    writer = None
    esquema = None
    total = invalidas = outros_anos = 0
    lote = []

    def gravar():
        nonlocal total, invalidas, outros_anos
        if not lote: return
        tabela = pa.Table.from_pylist(lote, schema=esquema)
        datas = datas_obito(pd.Series([r['DTOBITO'] for r in lote], dtype='string'))
        invalidas += int(datas.isna().sum())
        outros_anos += int((datas.notna() & datas.dt.year.ne(ano)).sum())
        writer.write_table(tabela)
        total += len(lote)
        lote.clear()
        print(f'  {ano}: {total:,} registros convertidos do XML', flush=True)

    try:
        with zipfile.ZipFile(origem) as z:
            membros = [n for n in z.namelist() if n.lower().endswith('.xml') and not n.startswith('__MACOSX/')]
            if not membros: raise ValueError('ZIP sem XML.')
            for nome in membros:
                with z.open(nome) as f:
                    eventos = ET.iterparse(f, events=('start', 'end'))
                    _, raiz = next(eventos)
                    if raiz.tag.lower() not in ('data', 'rows'): raise ValueError(f'Raiz XML inesperada: {raiz.tag}')
                    for evento, elem in eventos:
                        if evento != 'end' or elem.tag.lower() != 'row': continue
                        campos = [c.tag.upper() for c in elem]
                        if len(campos) != len(set(campos)): raise ValueError('XML com campos repetidos na mesma linha.')
                        if any(len(c) for c in elem): raise ValueError('XML com campos aninhados: layout nao suportado.')
                        registro = {c.tag.upper(): c.text or '' for c in elem}
                        if esquema is None:
                            if not {'DTOBITO', 'CAUSABAS'}.issubset(registro):
                                raise ValueError('XML sem DTOBITO/CAUSABAS nomeados. Conversao interrompida.')
                            esquema = pa.schema([(c, pa.string()) for c in campos])
                            writer = pq.ParquetWriter(parcial, esquema, compression='snappy')
                        elif set(campos) != set(esquema.names):
                            raise ValueError('Layout XML mudou entre registros. Conversao interrompida para preservar todas as colunas.')
                        lote.append(registro)
                        elem.clear()
                        raiz.clear()
                        if len(lote) >= LOTE: gravar()
            gravar()
        if writer is not None:
            writer.close()
            writer = None
        if total == 0: raise ValueError('XML sem registros.')
        if pq.read_metadata(parcial).num_rows != total: raise ValueError('Contagem Parquet divergente.')
        parcial.replace(destino)
        return {'registros': total, 'colunas': len(esquema), 'formato_origem': 'XML',
                'xml_no_zip': membros, 'datas_invalidas_ou_ausentes': invalidas,
                'datas_de_outro_ano': outros_anos}
    finally:
        if writer is not None: writer.close()


def main():
    global PASTA
    identificador = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
    PASTA = BASE / 'dados_mortalidade_sim' / ('coleta_' + identificador)
    PASTA.mkdir(parents=True, exist_ok=False)
    for sub in ('brutos', 'parquet'):
        (PASTA / sub).mkdir()
    manifesto = PASTA / 'manifesto_download.json'
    estados = {}
    falhas = []
    print('Nova coleta completa: ' + str(PASTA), flush=True)
    for ano in ANOS:
        url = URLS[ano]
        formato = 'xml' if ano in (2022, 2023) else 'csv'
        zip_path = PASTA / 'brutos' / f'sim_{ano}_{formato}.zip'
        parquet = PASTA / 'parquet' / f'sim_{ano}.parquet'
        print(f'\n{ano}: download {formato.upper()} e conversao para Parquet', flush=True)
        estados[str(ano)] = {'status': 'em_andamento', 'url': url, 'formato_origem': formato.upper()}
        salvar_json(manifesto, estados)
        try:
            baixar(url, zip_path)
            info = (converter_xml if formato == 'xml' else converter)(zip_path, parquet, ano)
            estados[str(ano)] = {
                'status': 'concluido', 'url': url, 'formato_origem': formato.upper(),
                'coletado_utc': datetime.now(timezone.utc).isoformat(),
                'parquet': str(parquet), 'zip': str(zip_path),
                'sha256_zip': sha256(zip_path), 'sha256_parquet': sha256(parquet), **info}
        except Exception as e:
            estados[str(ano)] = {'status': 'erro', 'url': url, 'erro': str(e)}
            falhas.append(ano)
            print(f'Falha em {ano}: {e}', flush=True)
        salvar_json(manifesto, estados)
    resumo = pd.DataFrame([{'ano': int(ano), **info} for ano, info in estados.items()])
    resumo.to_csv(PASTA / 'resumo_download.csv', index=False, sep=';', encoding='utf-8-sig')
    print('\nPasta desta execucao: ' + str(PASTA), flush=True)
    if falhas:
        raise RuntimeError(
            f'Coleta incompleta: {falhas}. Veja manifesto_download.json em {PASTA}. '
            'Uma nova execucao deste script baixa todos os anos novamente.')
    print('Concluido: cinco Parquets em ' + str(PASTA / 'parquet'), flush=True)
    print('ZIPs originais e contagens preservados. Nenhum filtro por causa foi aplicado.')
    return estados


if __name__ == '__main__':
    resultado_sim = main()


Nova coleta completa: d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_mortalidade_sim\coleta_20261023_103728_a8cccf84

2020: download CSV e conversao para Parquet
  Baixados 37.0 MB / 87.3 MB
  Baixados 77.0 MB / 87.3 MB
  2020: 50,000 registros convertidos
  2020: 100,000 registros convertidos
  2020: 150,000 registros convertidos
  2020: 200,000 registros convertidos
  2020: 250,000 registros convertidos
  2020: 300,000 registros convertidos
  2020: 350,000 registros convertidos
  2020: 400,000 registros convertidos
  2020: 450,000 registros convertidos
  2020: 500,000 registros convertidos
  2020: 550,000 registros convertidos
  2020: 600,000 registros convertidos
  2020: 650,000 registros convertidos
  2020: 700,000 registros convertidos
  2020: 750,000 registros convertidos
  2020: 800,000 registros convertidos
  2020: 850,000 registros convertidos
  2020: 900,000 registros convertidos
  2020: 950,000 registros convertidos
  2020: 1,000,000 registros convertidos
  2020

RuntimeError: Coleta incompleta: [2022]. Veja manifesto_download.json em d:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\dados_mortalidade_sim\coleta_20261023_103728_a8cccf84. Uma nova execucao deste script baixa todos os anos novamente.